# Train Test Creator

## Install libraries

In [25]:
import os
import sys
import random
from dotenv import load_dotenv
import pandas as pd

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from logger.logger import Logger
from utils.constants import *
from utils.utils import *
from tabular_database_driver.postgre_sql_driver import PostgreSQLDriver
from dtos.tabular_database_driver_dtos.postgre_sql_connection_dto import (
    PostgreSQLConnectionDto,
)
from dtos.tabular_database_driver_dtos.tabular_database_driver_dtos import *
from ta.ta_functions import *

load_dotenv()

True

In [14]:
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## Parameters

In [60]:
STOCK_CODE = "VIC"
LOOKBACK_WINDOW = 50
FORECAST_HORIZON = 5
TARGET_COLUMN = f"return_{FORECAST_HORIZON}"

# Inclusive
TRAIN_RANGE = ("2000-01-01", "2021-12-31")
VAL_RANGE = ("2022-01-01", "2023-12-31")
TEST_RANGE = ("2024-01-01", "2026-04-30")

In [16]:
STOCK_CODE = str.lower(STOCK_CODE)
STOCK_CODE

'vic'

## Load data

In [17]:
my_logger = Logger(
    file_name=f"{FEATURE_SELECTION_LOG_FILE_BASE}/{STOCK_CODE}/train_test_creator.log",
)

In [18]:
my_connection_model = PostgreSQLConnectionDto(
    logger=my_logger,
    host=os.getenv("POSTGRES_HOST"),
    user=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
    port=os.getenv("POSTGRES_PORT"),
    database=os.getenv("GOLD_POSTGRES_DATABASE"),
)

In [19]:
my_postgresql_driver = PostgreSQLDriver(logger=my_logger)
my_postgresql_driver.connect(my_connection_model)

<DatabaseExecutionStatus.SUCCESS: 'success'>

In [20]:
stock_df = my_postgresql_driver.select(
    schema_name=Schema.ENTERPRISE.value,
    table_name=f"unified_{STOCK_CODE}",
    order_by=["date"],
)

stock_df

,code,date,close,adjust,change,matching_volume,matching_value,negotiate_volume,negotiate_value,open,...,date_month_cos,date_dow_sin,date_dow_cos,date_hour_sin,date_hour_cos,date_quarter_sin,date_quarter_cos,date_doy_sin,date_doy_cos,date_unix_ts
0,VIC,2007-09-19,125.0,2.43,0.0,307840,38.48,0,0.0,125.0,...,-0.00000000000000018369701987210297,0.9749279121818236,-0.22252093395631434,0.0,1.0,-1.0,-0.00000000000000018369701987210297,-0.9796136916454901,-0.20089055513063528,1190160000
1,VIC,2007-09-20,131.0,2.55,6.0,794790,104.12,0,0.0,131.0,...,-0.00000000000000018369701987210297,0.43388373911755823,-0.900968867902419,0.0,1.0,-1.0,-0.00000000000000018369701987210297,-0.9829265519799822,-0.18399835165767986,1190246400
2,VIC,2007-09-21,137.0,2.66,6.0,1224660,167.4,0,0.0,137.0,...,-0.00000000000000018369701987210297,-0.433883739117558,-0.9009688679024191,0.0,1.0,-1.0,-0.00000000000000018369701987210297,-0.9859481499638303,-0.1670516255021195,1190332800
3,VIC,2007-09-24,143.0,2.78,6.0,551130,78.81,0,0.0,143.0,...,-0.00000000000000018369701987210297,0.0,1.0,0.0,1.0,-1.0,-0.00000000000000018369701987210297,-0.9932568492674143,-0.11593459959550066,1190592000
4,VIC,2007-09-25,150.0,2.92,7.0,962110,144.3,0,0.0,150.0,...,-0.00000000000000018369701987210297,0.7818314824680298,0.6234898018587336,0.0,1.0,-1.0,-0.00000000000000018369701987210297,-0.9951053111006974,-0.09882013873287211,1190678400
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4567,VIC,2026-04-22,207.2,207.2,13.5,4713700,944.3,0,0.0,193.6,...,-0.4999999999999998,0.9749279121818236,-0.22252093395631434,0.0,1.0,0.00000000000000012246467991473532,-1.0,0.9368813462954315,-0.3496474552512284,1776816000
4568,VIC,2026-04-23,214.5,214.5,7.3,4258100,910.55,0,0.0,212.0,...,-0.4999999999999998,0.43388373911755823,-0.900968867902419,0.0,1.0,0.00000000000000012246467991473532,-1.0,0.9307239310379795,-0.36572252349726897,1776902400
4569,VIC,2026-04-24,212.1,212.1,-2.4,4235200,909.84,0,0.0,215.2,...,-0.4999999999999998,-0.433883739117558,-0.9009688679024191,0.0,1.0,0.00000000000000012246467991473532,-1.0,0.9242907221930933,-0.3816892202666588,1776988800
4570,VIC,2026-04-28,225.5,225.5,13.4,5194900,1159.3,0,0.0,210.0,...,-0.4999999999999998,0.7818314824680298,0.6234898018587336,0.0,1.0,0.00000000000000012246467991473532,-1.0,0.895839290734909,-0.4443781781046132,1777334400


## Split Train Val Test

In [21]:
train_df = stock_df[
    (stock_df["date"] >= TRAIN_RANGE[0]) & (stock_df["date"] <= TRAIN_RANGE[1])
]
train_df

,code,date,close,adjust,change,matching_volume,matching_value,negotiate_volume,negotiate_value,open,...,date_month_cos,date_dow_sin,date_dow_cos,date_hour_sin,date_hour_cos,date_quarter_sin,date_quarter_cos,date_doy_sin,date_doy_cos,date_unix_ts
0,VIC,2007-09-19,125.0,2.43,0.0,307840,38.48,0,0.0,125.0,...,-0.00000000000000018369701987210297,0.9749279121818236,-0.22252093395631434,0.0,1.0,-1.0,-0.00000000000000018369701987210297,-0.9796136916454901,-0.20089055513063528,1190160000
1,VIC,2007-09-20,131.0,2.55,6.0,794790,104.12,0,0.0,131.0,...,-0.00000000000000018369701987210297,0.43388373911755823,-0.900968867902419,0.0,1.0,-1.0,-0.00000000000000018369701987210297,-0.9829265519799822,-0.18399835165767986,1190246400
2,VIC,2007-09-21,137.0,2.66,6.0,1224660,167.4,0,0.0,137.0,...,-0.00000000000000018369701987210297,-0.433883739117558,-0.9009688679024191,0.0,1.0,-1.0,-0.00000000000000018369701987210297,-0.9859481499638303,-0.1670516255021195,1190332800
3,VIC,2007-09-24,143.0,2.78,6.0,551130,78.81,0,0.0,143.0,...,-0.00000000000000018369701987210297,0.0,1.0,0.0,1.0,-1.0,-0.00000000000000018369701987210297,-0.9932568492674143,-0.11593459959550066,1190592000
4,VIC,2007-09-25,150.0,2.92,7.0,962110,144.3,0,0.0,150.0,...,-0.00000000000000018369701987210297,0.7818314824680298,0.6234898018587336,0.0,1.0,-1.0,-0.00000000000000018369701987210297,-0.9951053111006974,-0.09882013873287211,1190678400
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3510,VIC,2021-12-24,96.5,48.25,0.5,1415500,136.06,0,0.0,96.5,...,1.0,-0.433883739117558,-0.9009688679024191,0.0,1.0,-0.00000000000000024492935982947064,1.0,-0.12020804489935275,0.9927487224577402,1640304000
3511,VIC,2021-12-27,99.0,49.5,2.5,1907500,186.16,0,0.0,97.0,...,1.0,0.0,1.0,0.0,1.0,-0.00000000000000024492935982947064,1.0,-0.06880242680232064,0.9976303053065857,1640563200
3512,VIC,2021-12-28,98.4,49.2,-0.6,1737300,169.92,0,0.0,99.1,...,1.0,0.7818314824680298,0.6234898018587336,0.0,1.0,-0.00000000000000024492935982947064,1.0,-0.05161966722325418,0.9986668162884759,1640649600
3513,VIC,2021-12-29,95.5,47.75,-2.9,2291900,220.49,0,0.0,98.0,...,1.0,0.9749279121818236,-0.22252093395631434,0.0,1.0,-0.00000000000000024492935982947064,1.0,-0.034421611622745804,0.9994074007397048,1640736000


In [22]:
val_df = stock_df[
    (stock_df["date"] >= VAL_RANGE[0]) & (stock_df["date"] <= VAL_RANGE[1])
]
val_df

,code,date,close,adjust,change,matching_volume,matching_value,negotiate_volume,negotiate_value,open,...,date_month_cos,date_dow_sin,date_dow_cos,date_hour_sin,date_hour_cos,date_quarter_sin,date_quarter_cos,date_doy_sin,date_doy_cos,date_unix_ts
3515,VIC,2022-01-04,101.0,50.5,5.9,3071100,303.1,0,0.0,96.0,...,0.8660254037844387,0.7818314824680298,0.6234898018587336,0.0,1.0,1.0,0.00000000000000006123233995736766,0.06880242680231986,0.9976303053065857,1641254400
3516,VIC,2022-01-05,100.0,50.0,-1.0,3396500,342.98,0,0.0,100.8,...,0.8660254037844387,0.9749279121818236,-0.22252093395631434,0.0,1.0,1.0,0.00000000000000006123233995736766,0.08596479873744647,0.9962981749346078,1641340800
3517,VIC,2022-01-06,104.5,52.25,4.5,5061400,531.06,0,0.0,101.0,...,0.8660254037844387,0.43388373911755823,-0.900968867902419,0.0,1.0,1.0,0.00000000000000006123233995736766,0.10310169744743485,0.9946708199115211,1641427200
3518,VIC,2022-01-07,102.2,51.1,-2.3,3108800,321.55,0,0.0,106.4,...,0.8660254037844387,-0.433883739117558,-0.9009688679024191,0.0,1.0,1.0,0.00000000000000006123233995736766,0.1202080448993527,0.9927487224577402,1641513600
3519,VIC,2022-01-10,102.3,51.15,0.1,2908500,302.2,0,0.0,102.5,...,0.8660254037844387,0.0,1.0,0.0,1.0,1.0,0.00000000000000006123233995736766,0.1712931441814776,0.9852201067560606,1641772800
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4003,VIC,2023-12-25,43.4,21.7,0.25,1977500,85.71,0,0.0,43.1,...,1.0,0.0,1.0,0.0,1.0,-0.00000000000000024492935982947064,1.0,-0.10310169744743544,0.9946708199115211,1703462400
4004,VIC,2023-12-26,43.55,21.78,0.15,1763700,76.82,0,0.0,43.4,...,1.0,0.7818314824680298,0.6234898018587336,0.0,1.0,-0.00000000000000024492935982947064,1.0,-0.0859647987374467,0.9962981749346077,1703548800
4005,VIC,2023-12-27,43.6,21.8,0.05,1848500,80.88,0,0.0,43.65,...,1.0,0.9749279121818236,-0.22252093395631434,0.0,1.0,-0.00000000000000024492935982947064,1.0,-0.06880242680232064,0.9976303053065857,1703635200
4006,VIC,2023-12-28,44.45,22.22,0.85,4070700,180.42,0,0.0,43.6,...,1.0,0.43388373911755823,-0.900968867902419,0.0,1.0,-0.00000000000000024492935982947064,1.0,-0.05161966722325418,0.9986668162884759,1703721600


In [23]:
test_df = stock_df[
    (stock_df["date"] >= TEST_RANGE[0]) & (stock_df["date"] <= TEST_RANGE[1])
]
test_df

,code,date,close,adjust,change,matching_volume,matching_value,negotiate_volume,negotiate_value,open,...,date_month_cos,date_dow_sin,date_dow_cos,date_hour_sin,date_hour_cos,date_quarter_sin,date_quarter_cos,date_doy_sin,date_doy_cos,date_unix_ts
4008,VIC,2024-01-02,44.0,22.0,-0.6,2281300,101.17,0,0.0,44.95,...,0.8660254037844387,0.7818314824680298,0.6234898018587336,0.0,1.0,1.0,0.00000000000000006123233995736766,0.034327600513243496,0.9994106342455052,1704153600
4009,VIC,2024-01-03,44.15,22.08,0.15,2275100,99.49,0,0.0,43.5,...,0.8660254037844387,0.9749279121818236,-0.22252093395631434,0.0,1.0,1.0,0.00000000000000006123233995736766,0.05147875477034653,0.9986740898848305,1704240000
4010,VIC,2024-01-04,44.15,22.08,0.0,2337800,103.19,0,0.0,44.15,...,0.8660254037844387,0.43388373911755823,-0.900968867902419,0.0,1.0,1.0,0.00000000000000006123233995736766,0.06861473800213404,0.9976432316860063,1704326400
4011,VIC,2024-01-05,44.1,22.05,-0.05,1481600,65.23,0,0.0,44.15,...,0.8660254037844387,-0.433883739117558,-0.9009688679024191,0.0,1.0,1.0,0.00000000000000006123233995736766,0.08573050015569435,0.9963183634476755,1704412800
4012,VIC,2024-01-08,44.35,22.18,0.25,2534400,112.6,0,0.0,44.45,...,0.8660254037844387,0.0,1.0,0.0,1.0,1.0,0.00000000000000006123233995736766,0.13690605792347524,0.990584035457797,1704672000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4567,VIC,2026-04-22,207.2,207.2,13.5,4713700,944.3,0,0.0,193.6,...,-0.4999999999999998,0.9749279121818236,-0.22252093395631434,0.0,1.0,0.00000000000000012246467991473532,-1.0,0.9368813462954315,-0.3496474552512284,1776816000
4568,VIC,2026-04-23,214.5,214.5,7.3,4258100,910.55,0,0.0,212.0,...,-0.4999999999999998,0.43388373911755823,-0.900968867902419,0.0,1.0,0.00000000000000012246467991473532,-1.0,0.9307239310379795,-0.36572252349726897,1776902400
4569,VIC,2026-04-24,212.1,212.1,-2.4,4235200,909.84,0,0.0,215.2,...,-0.4999999999999998,-0.433883739117558,-0.9009688679024191,0.0,1.0,0.00000000000000012246467991473532,-1.0,0.9242907221930933,-0.3816892202666588,1776988800
4570,VIC,2026-04-28,225.5,225.5,13.4,5194900,1159.3,0,0.0,210.0,...,-0.4999999999999998,0.7818314824680298,0.6234898018587336,0.0,1.0,0.00000000000000012246467991473532,-1.0,0.895839290734909,-0.4443781781046132,1777334400


## Create features

In [47]:
feature_functions = [
    lambda df: add_ad(df, n=[10]),
]
len(feature_functions)

1

In [48]:
def apply_features(df, funcs):
    for func in funcs:
        df = func(df)
    return df


featured_train_df = apply_features(train_df, feature_functions)
featured_val_df = apply_features(val_df, feature_functions)
featured_test_df = apply_features(test_df, feature_functions)

In [49]:
featured_train_df

,code,date,close,adjust,change,matching_volume,matching_value,negotiate_volume,negotiate_value,open,...,ad_direction,ad_signal_10,ad_signal_10_slope,ad_hist_10,ad_hist_10_slope,ad_hist_10_acceleration,ad_hist_10_gt_0,ad_hist_10_lt_0,ad_hist_10_abs,ad_10_strength
0,VIC,2007-09-19,125.0,2.43,0.0,307840,38.48,0,0.0,125.0,...,-1,0.000000e+00,NaN,0.000000e+00,NaN,NaN,False,False,0.000000e+00,NaN
1,VIC,2007-09-20,131.0,2.55,6.0,794790,104.12,0,0.0,131.0,...,1,1.445073e+05,144507.272727,6.502827e+05,6.502827e+05,NaN,True,False,6.502827e+05,5.168382e+11
2,VIC,2007-09-21,137.0,2.66,6.0,1224660,167.4,0,0.0,137.0,...,1,4.854060e+05,340898.677686,1.534044e+06,8.837613e+05,2.334786e+05,True,False,1.534044e+06,1.878682e+12
3,VIC,2007-09-24,143.0,2.78,6.0,551130,78.81,0,0.0,143.0,...,-1,7.643231e+05,278917.099925,1.255127e+06,-2.789171e+05,-1.162678e+06,True,False,1.255127e+06,0.000000e+00
4,VIC,2007-09-25,150.0,2.92,7.0,962110,144.3,0,0.0,150.0,...,1,1.167457e+06,403133.990848,1.814103e+06,5.589760e+05,8.378931e+05,True,False,1.814103e+06,1.745367e+12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3510,VIC,2021-12-24,96.5,48.25,0.5,1415500,136.06,0,0.0,96.5,...,1,9.796396e+07,-157147.745926,-7.071649e+05,9.435366e+05,3.347814e+06,False,True,7.071649e+05,5.561066e+11
3511,VIC,2021-12-27,99.0,49.5,2.5,1907500,186.16,0,0.0,97.0,...,1,9.818220e+07,218242.753334,9.820924e+05,1.689257e+06,7.457206e+05,True,False,9.820924e+05,1.873341e+12
3512,VIC,2021-12-28,98.4,49.2,-0.6,1737300,169.92,0,0.0,99.1,...,1,9.847358e+07,291373.941039,1.311183e+06,3.290903e+05,-1.360167e+06,True,False,1.311183e+06,8.135421e+11
3513,VIC,2021-12-29,95.5,47.75,-2.9,2291900,220.49,0,0.0,98.0,...,-1,9.838456e+07,-89017.424864,-4.005784e+05,-1.711761e+06,-2.040851e+06,False,True,4.005784e+05,7.213530e+11


In [50]:
featured_val_df

,code,date,close,adjust,change,matching_volume,matching_value,negotiate_volume,negotiate_value,open,...,ad_direction,ad_signal_10,ad_signal_10_slope,ad_hist_10,ad_hist_10_slope,ad_hist_10_acceleration,ad_hist_10_gt_0,ad_hist_10_lt_0,ad_hist_10_abs,ad_10_strength
3515,VIC,2022-01-04,101.0,50.5,5.9,3071100,303.1,0,0.0,96.0,...,-1,2.541600e+06,NaN,0.000000e+00,NaN,NaN,False,False,0.000000e+00,NaN
3516,VIC,2022-01-05,100.0,50.0,-1.0,3396500,342.98,0,0.0,100.8,...,-1,2.152775e+06,-388824.915825,-1.749712e+06,-1.749712e+06,NaN,False,True,1.749712e+06,3.741824e+12
3517,VIC,2022-01-06,104.5,52.25,4.5,5061400,531.06,0,0.0,101.0,...,1,2.162194e+06,9418.751459,4.238438e+04,1.792097e+06,3.541809e+06,True,False,4.238438e+04,7.635611e+10
3518,VIC,2022-01-07,102.2,51.1,-2.3,3108800,321.55,0,0.0,106.4,...,-1,1.604664e+06,-557530.112443,-2.508886e+06,-2.551270e+06,-4.343366e+06,False,True,2.508886e+06,7.799623e+12
3519,VIC,2022-01-10,102.3,51.15,0.1,2908500,302.2,0,0.0,102.5,...,-1,6.517341e+05,-952929.596131,-4.288183e+06,-1.779298e+06,7.719722e+05,False,True,4.288183e+06,1.171629e+13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4003,VIC,2023-12-25,43.4,21.7,0.25,1977500,85.71,0,0.0,43.1,...,1,-7.623207e+07,897240.733865,4.037583e+06,1.622902e+03,2.314697e+05,True,False,4.037583e+06,3.629237e+12
4004,VIC,2023-12-26,43.55,21.78,0.15,1763700,76.82,0,0.0,43.4,...,-1,-7.549797e+07,734106.054980,3.303477e+06,-7.341061e+05,-7.357290e+05,True,False,3.303477e+06,9.845129e-02
4005,VIC,2023-12-27,43.6,21.8,0.05,1848500,80.88,0,0.0,43.65,...,-1,-7.523343e+07,264541.317711,1.190436e+06,-2.113041e+06,-1.378935e+06,True,False,1.190436e+06,2.200521e+12
4006,VIC,2023-12-28,44.45,22.22,0.85,4070700,180.42,0,0.0,43.6,...,1,-7.449890e+07,734531.987218,3.305394e+06,2.114958e+06,4.227999e+06,True,False,3.305394e+06,9.418687e+12


In [51]:
featured_test_df

,code,date,close,adjust,change,matching_volume,matching_value,negotiate_volume,negotiate_value,open,...,ad_direction,ad_signal_10,ad_signal_10_slope,ad_hist_10,ad_hist_10_slope,ad_hist_10_acceleration,ad_hist_10_gt_0,ad_hist_10_lt_0,ad_hist_10_abs,ad_10_strength
4008,VIC,2024-01-02,44.0,22.0,-0.6,2281300,101.17,0,0.0,44.95,...,-1,-2.281300e+06,NaN,0.000000e+00,NaN,NaN,False,False,0.000000e+00,NaN
4009,VIC,2024-01-03,44.15,22.08,0.15,2275100,99.49,0,0.0,43.5,...,1,-1.867645e+06,4.136545e+05,1.861445e+06,1.861445e+06,NaN,True,False,1.861445e+06,4.234975e+12
4010,VIC,2024-01-04,44.15,22.08,0.0,2337800,103.19,0,0.0,44.15,...,1,-1.458358e+06,4.092871e+05,1.841792e+06,-1.965372e+04,-1.881099e+06,True,False,1.841792e+06,7.176235e+11
4011,VIC,2024-01-05,44.1,22.05,-0.05,1481600,65.23,0,0.0,44.15,...,1,-1.033693e+06,4.246652e+05,1.910993e+06,6.920150e+04,8.885522e+04,True,False,1.910993e+06,9.437759e+11
4012,VIC,2024-01-08,44.35,22.18,0.25,2534400,112.6,0,0.0,44.45,...,-1,-7.925784e+05,2.411149e+05,1.085017e+06,-8.259764e+05,-8.951779e+05,True,False,1.085017e+06,6.345846e+11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4567,VIC,2026-04-22,207.2,207.2,13.5,4713700,944.3,0,0.0,193.6,...,1,7.396464e+07,1.753588e+06,7.891147e+06,2.960112e+06,6.482301e+06,True,False,7.891147e+06,3.719650e+13
4568,VIC,2026-04-23,214.5,214.5,7.3,4258100,910.55,0,0.0,212.0,...,1,7.547839e+07,1.513754e+06,6.811893e+06,-1.079254e+06,-4.039366e+06,True,False,6.811893e+06,2.959768e+12
4569,VIC,2026-04-24,212.1,212.1,-2.4,4235200,909.84,0,0.0,215.2,...,-1,7.640115e+07,9.227558e+05,4.152401e+06,-2.659492e+06,-1.580238e+06,True,False,4.152401e+06,7.211627e+12
4570,VIC,2026-04-28,225.5,225.5,13.4,5194900,1159.3,0,0.0,210.0,...,1,7.794600e+07,1.544850e+06,6.951824e+06,2.799423e+06,5.458916e+06,True,False,6.951824e+06,3.020062e+13


## Standardization

In [58]:
def categorize_columns(df, ordinal_map: dict = None):
    """
    Auto-cast columns to suitable dtypes, then categorize into 3 lists.

    Parameters
    ----------
    df : pd.DataFrame
    ordinal_map : dict, optional
        {col_name: [ordered_categories]} for columns that should be ordinal.
        Example: {"size": ["S", "M", "L"], "priority": ["low", "med", "high"]}

    Returns
    -------
    numerical, nominal_categorical, ordinal_categorical : list of column names
    """
    ordinal_map = ordinal_map or {}
    df = df.copy()

    for col in df.columns:
        # --- 1. Try casting object/string columns ---
        if df[col].dtype == object or isinstance(df[col].dtype, pd.StringDtype):
            # Try numeric first
            converted = pd.to_numeric(df[col], errors="coerce")
            if converted.notna().sum() / len(df) >= 0.9:  # 90%+ parseable → numeric
                df[col] = converted
            else:
                # Fall through to categorical casting below
                pass

        # --- 2. Cast to ordinal categorical ---
        if col in ordinal_map:
            df[col] = pd.Categorical(df[col], categories=ordinal_map[col], ordered=True)

        # --- 3. Cast remaining object/string → nominal categorical ---
        elif df[col].dtype == object or isinstance(df[col].dtype, pd.StringDtype):
            df[col] = pd.Categorical(df[col])

    # --- 4. Categorize ---
    numerical, nominal_categorical, ordinal_categorical = [], [], []

    for col in df.columns:
        dtype = df[col].dtype
        if pd.api.types.is_numeric_dtype(dtype):
            numerical.append(col)
        elif isinstance(dtype, pd.CategoricalDtype):
            if dtype.ordered:
                ordinal_categorical.append(col)
            else:
                nominal_categorical.append(col)

    return numerical, nominal_categorical, ordinal_categorical

In [59]:
numerical, nominal, ordinal = categorize_columns(featured_train_df)
numerical, nominal, ordinal

(['close',
  'adjust',
  'change',
  'matching_volume',
  'matching_value',
  'negotiate_volume',
  'negotiate_value',
  'open',
  'high',
  'low',
  'percent_change',
  'number_of_buy_orders',
  'buy_volume',
  'average_volume_per_buy_order',
  'number_of_sell_orders',
  'sell_volume',
  'average_volume_per_sell_order',
  'net_volume',
  'date_year',
  'date_month',
  'date_day',
  'date_hour',
  'date_minute',
  'date_second',
  'date_week',
  'date_day_of_week',
  'date_day_of_year',
  'date_quarter',
  'date_day_of_quarter',
  'date_days_to_quarter_end',
  'date_quarter_progress',
  'date_days_in_month',
  'date_days_to_month_end',
  'date_week_of_month',
  'date_days_in_year',
  'date_days_to_year_end',
  'date_year_progress',
  'date_is_leap_year',
  'date_is_weekend',
  'date_is_weekday',
  'date_is_month_start',
  'date_is_month_end',
  'date_is_quarter_start',
  'date_is_quarter_end',
  'date_is_year_start',
  'date_is_year_end',
  'date_month_sin',
  'date_month_cos',
  'date